# **LLM Prompting**

## What is Prompt?

In the context of LLMs, a prompt is the input text that you provide to the model to generate a response. It can be a question, a statement, or any piece of text that you want the model to process and respond to. The quality and clarity of the prompt can significantly affect the quality of the response generated by the LLM. Therefore, crafting effective prompts is an important skill when working with LLMs.

To understand why we need prompt, we need to understand how LLMs work. LLMs, and by general definition a generative model, are model that trained to predict the next token in a sequence of text. Thats why we need to provide a prompt, this is the initial sequence of text that the model will use to generate the next token. The model will then use the prompt to generate a response based on the patterns it has learned from the training data.

### Its All About the Context

Because of the way LLMs works, the prompt, or we call it 'context', is crucial for the model to generate a relevant and accurate response. The model relies on the context provided in the prompt to understand what is being asked and to generate a response that is coherent and relevant to the input. If the prompt is vague or lacks sufficient context, the model may generate a response that is off-topic or not useful. Therefore, providing clear and detailed context in the prompt can help improve the quality of the response generated by the LLM.

The context can include information about the topic, the desired format of the response, or any specific instructions that you want the model to follow. For example, if you want the model to generate a summary of a text, you can provide the text as context and ask the model to summarize it. The more specific and detailed the context is, the better the model can understand what you are asking for and generate a relevant response.

Because of this, the task we expect the model to do (e.g summarization, question answering, etc) is not determined by the model itself, but rather by the context we provide in the prompt. This is why crafting effective prompts is crucial when working with LLMs, as it can significantly impact the quality of the responses generated by the model.

### Structuring the Prompt

When constructing a prompt, it's important to consider the structure and format of the input text. A well-structured prompt can help the model understand the context and generate a more accurate response. Here are some tips for structuring your prompts effectively:

1. **Be Clear and Specific**: Clearly state what you want the model to do. Avoid vague language and provide specific instructions if necessary.
2. **Provide Context**: Include relevant information that can help the model understand the task. This could be background information, examples, or any other details that are pertinent to the request.
3. **Use Proper Formatting**: If you want the model to generate a response in a specific format (e.g., a list, a table, etc.), make sure to indicate that in the prompt.
4. **Break Down Complex Tasks**: If the task is complex, consider breaking it down into smaller, more manageable parts. This can help the model focus on one aspect of the task at a time and generate a more accurate response.

For example, if you want the model to generate a summary of a text, you could structure your prompt like this:

```
Summarize the following text. Format the summary as paragraphs and include the main points. Do not include any personal opinions or interpretations.

[Insert text here]
```

## Interacting with the Model

LLMs are capable of a vast range of tasks due to inherent property we call 'reasoning.' While the term suggests the model is 'thinking' in a human sense, it is actually generating text by navigating complex patterns learned during training. Although the model lacks true consciousness or understanding, it produces responses that appear thoughtful and coherent based on the input it receives. Despite this distinction, this functional reasoning remains the core of an LLM's capability.

In recent advancements, this reasoning capability has been further enhanced by techniques like [Inference-Time Scaling](https://magazine.sebastianraschka.com/p/categories-of-inference-time-scaling). This process, though invisible to the user, allows the model to 'think' through a problem before responding, significantly improving its accuracy and performance.

This approach creates model that some call 'reasoning models,' which are designed to perform better on tasks that require logical thinking and problem-solving. By allowing the model to take more time to process the input and generate a response, it can produce more accurate and relevant answers, especially for complex queries. This allows users to match the model’s level of analysis to their specific use case, whether it is a simple question or a complex task requiring deep logical breakdown.

### Adjusting Parameters

In programming aspect, we can control the model's behavior by adjusting parameters such as temperature and the aforementioned 'reasoning effort.'

The temperature parameter controls the randomness of the model's output. A higher temperature will result in more creative and diverse responses, while a lower temperature will produce more focused and deterministic responses. By adjusting the temperature, you can influence the style and creativity of the model's output.

The 'reasoning effort' parameter, on the other hand, allows you to control how much time the model spends processing the input before generating a response. By increasing the reasoning effort, it allow the model to take more time to generate a response, but on the other hand, the cost of using the model will also increase.

The currency we use in this case is 'tokens,' which are the basic units of text that the model processes. The more tokens the model generates, the higher the cost. Therefore, it's important to find a balance between the quality of the response and the cost of using the model by adjusting these parameters accordingly.

Example of adjusting the temperature and reasoning effort parameters:

```python
result = client.chat.completions.parse(
    ...,
    temperature=0.7,  # Adjust the creativity of the response
    reasoning_effort=None  # Disable reasoning effort
)
```

In [ ]:
!pip install openai

In [1]:
from openai import OpenAI

model = "liquid/lfm-2.5-1.2b-instruct:free"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="<OPENROUTER_API_KEY>",
)

result = client.chat.completions.parse(
    model=model,
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?",
        }
    ],
    temperature=0.7,  # Adjust the creativity of the response
    reasoning_effort=None,  # Disable reasoning effort
)

print(result.choices[0].message.content)

The capital of France is **Paris**.


### Output Schema Validation

Some advanced concept in the LLM usage is Schema Validation, which is a technique to ensure that the output generated by the LLM adheres to a specific format or structure. This is particularly useful when you want to ensure that the output can be easily parsed and used in downstream applications.

We can use [pydantic](https://pydantic.dev/) to define the expected output schema and validate the output generated by the LLM. This allows us to catch any errors or inconsistencies in the output early on and handle them appropriately.

In [2]:
from typing import Literal
from pydantic import BaseModel


# Literal of "truth" or "fake"
class TruthOrFake(BaseModel):
    answer: Literal["truth", "fake"]


result = client.chat.completions.parse(
    model=model,
    messages=[
        {
            "role": "user",
            "content": "Blueberry color is actually purple.",
        },
    ],
    response_format=TruthOrFake,
)

parsed = result.choices[0].message.parsed
print(parsed.answer)

truth


With this technique, we can also do something like information extraction, where we can extract specific pieces of information from the LLM's output based on the defined schema. This can be particularly useful for tasks like data extraction, where you want to extract specific fields from a larger body of text.

In [6]:
class Employee(BaseModel):
    name: str
    age: int


class EmployeeList(BaseModel):
    employees: list[Employee]


result = client.chat.completions.parse(
    model=model,
    messages=[
        {
            "role": "user",
            "content": "The Product Team consists of Alice(30) and Bob(25).",
        }
    ],
    response_format=EmployeeList,
)

parsed = result.choices[0].message.parsed
for employee in parsed.employees:
    print(f"Name: {employee.name}, Age: {employee.age}")

Name: Alice, Age: 30
Name: Bob, Age: 25


For further reading, you can check the pydantic [documentation on LLM](https://pydantic.dev/articles/llm-intro).